# Test generation: requirements to declarations

Choose a small shape domain, generate coverage points, and inspect real capsule
declarations, interface MLIR and integer reference outputs. All sample dimensions
and arithmetic are illustrative. No Torch or hardware is needed; results stay in memory.

Use the [shared tutorial launcher](README.md) to set up and open the notebooks.
[The walkthrough](../../docs/guides/10-compiler-generation.md) explains the full Phase 0
route and the command for a target with selected software and hardware inputs.


**Step 2 of 5: Test generation** | [All tutorials](README.md) | [Previous: Contracts and facts](01_contracts.ipynb) | [Next: Scheduling](03_scheduling.ipynb)


In [ ]:
import json

from merlin.development.testgen.component_coverage_inputs import covering_points
from merlin.development.testgen.component_coverage_plan import COHORTS
from merlin.targetgen.capsule_golden import golden
from merlin.targetgen.corpus_spec import CorpusBinding, build

# Explicit source dimensions, not values inferred from hardware geometry.
ILLUSTRATIVE_AXES = {"M": [2, 3], "K": [2, 4], "N": [2, 3]}
ILLUSTRATIVE_BINDING = CorpusBinding(
    target="illustrative",
    tile_dim=None,
    operand_dtype="int8",
    accum_dtype="i32",
    integer=True,
    tiers=[],
    compare="exact_int",
    readout_inputs={},
)


## 1. Cover the shape domain

`covering_points` is Phase 0's actual coverage algorithm. It covers each axis and
every pair of axis values, plus any larger interaction you request. With two
values on each of three axes, there are 18 single-axis/pair cells. The current
algorithm covers them with five points. A smaller member budget reports the
remaining cells rather than silently dropping requirements.


In [ ]:
def coverage_preview(member_budget=8, three_way=False):
    interactions = [["M", "K", "N"]] if three_way else []
    return covering_points(
        ILLUSTRATIVE_AXES,
        interactions,
        max_members=member_budget,
        max_cells=100,
    )


points, coverage = coverage_preview()
print(json.dumps({"points": points, "coverage": coverage}, indent=2))
assert coverage["status"] == "complete"
assert coverage["required_cells"] == 18 and len(points) == 5

_, limited = coverage_preview(member_budget=1)
print("Budget of one:", limited)
assert limited["status"] == "incomplete" and limited["missing_cells"] > 0


## 2. Build a declaration and interface program

`corpus_spec.build` turns an entry and binding into a capsule mapping and
`merlin_iface` program. Here we request explicit `M/K/N`, signed inputs and i32
accumulation. Inspect the operand dimensions and matmul/commit operations below.

`tile_dim=None` supplies no hardware geometry. The target name is illustrative;
a real target binding comes from its selected software declaration, facts and
capability contract. `merlin_iface` is the semantic input to lowering; it is not
yet OOT or device code.


In [ ]:
def declaration_preview(point, ordinal=0):
    entry = {
        "name": f"illustrative_matmul_{ordinal:02d}",
        "op": "matmul",
        "kind": "layer",
        "source_role": "illustrative_declaration",
        "source_reference": "Tutorial inputs; not hardware evidence or an admitted corpus",
        "label": "public",
        "output_dtype": "i32",
        "stimulus_range": [-2, 2],
        **point,
    }
    return build(entry, ILLUSTRATIVE_BINDING)


capsule, interface_mlir = declaration_preview(points[0])
print(json.dumps(capsule, indent=2))
print(interface_mlir)
assert capsule["expected"]["instruction_classes"] == []
assert capsule["required_oracle_tiers"] == []


## 3. Compute expected outputs

`capsule_golden.golden` evaluates the declaration on Merlin's integer Tensor
reference and deterministic inputs. It uses the declared operation rather than a
candidate's command buffer. These are expected values for later comparison.

The second call shows a missing builder being refused instead of generating a
different operation.


In [ ]:
reference_outputs = golden(capsule)
print("Expected host-reference outputs:")
print(json.dumps(reference_outputs, indent=2))
assert set(reference_outputs) == {"Y0"}
assert len(reference_outputs["Y0"]) == points[0]["M"]

try:
    build({"op": "unimplemented_tutorial_operation"}, ILLUSTRATIVE_BINDING)
except ValueError as error:
    print("Missing builder remains a refusal:", error)
else:
    raise AssertionError("an unsupported operation was silently generated")


## 4. Explore the coverage budget

Require the full `M/K/N` interaction to cover all eight shapes. Lower the member
budget and inspect covered/missing cells. The point menu follows the generated
points, so each selection shows an actual declaration, program and expected values.

Without widgets, call `show_preview(member_budget=3, three_way=True)` directly.


In [ ]:
from html import escape

try:
    import ipywidgets as widgets
    from IPython.display import HTML, clear_output, display
except ImportError:
    widgets = None
    HTML = None


def preview_html(points, report, selected, declaration, source, expected):
    required = report["required_cells"]
    missing = report["missing_cells"]
    covered = required - missing
    color = "#18734b" if report["status"] == "complete" else "#a55b08"
    rows = []
    for index, point in enumerate(points):
        style = ' style="background:#e9f4ff;font-weight:600"' if index == selected else ""
        values = [index, point["M"], point["K"], point["N"]]
        rows.append(f"<tr{style}>" + "".join(f"<td>{value}</td>" for value in values) + "</tr>")
    return (
        f'<div style="border-left:4px solid {color};padding:10px 14px">'
        f'<b>{escape(report["status"].upper())}</b> | '
        f"Covered {covered}/{required} cells | Missing {missing} | Members {len(points)}<br>"
        f'<progress value="{covered}" max="{required}" style="width:100%"></progress></div>'
        '<table style="margin:12px 0;border-spacing:18px 4px">'
        "<thead><tr><th>Point</th><th>M</th><th>K</th><th>N</th></tr></thead>"
        "<tbody>" + "".join(rows) + "</tbody></table>"
        f"<h4>Selected program: {escape(declaration['name'])}</h4>"
        f"<pre>{escape(source)}</pre>"
        f"<h4>Expected integer outputs</h4><pre>{escape(json.dumps(expected, indent=2))}</pre>"
        "<details><summary>Full generated declaration</summary>"
        f"<pre>{escape(json.dumps(declaration, indent=2))}</pre></details>"
    )


def show_preview(member_budget=8, three_way=False, point_index=0):
    selected_points, report = coverage_preview(member_budget, three_way)
    if not 0 <= point_index < len(selected_points):
        print(f"Choose a point index from 0 to {len(selected_points) - 1}.")
        return
    declaration, source = declaration_preview(selected_points[point_index], point_index)
    expected = golden(declaration)
    if HTML is not None:
        display(HTML(preview_html(selected_points, report, point_index, declaration, source, expected)))
    else:
        print("Finite-domain coverage:", json.dumps(report, sort_keys=True))
        for index, point in enumerate(selected_points):
            print(f"Point {index}: M={point['M']}, K={point['K']}, N={point['N']}")
        print("Expected integer outputs:", json.dumps(expected))
        print(source)


def coverage_widget():
    budget = widgets.IntSlider(value=8, min=1, max=8, description="Members", continuous_update=False)
    three_way = widgets.Checkbox(value=False, description="Require M/K/N")
    point = widgets.Dropdown(description="Point")
    output = widgets.Output()
    refreshing = False

    def render(change=None):
        if refreshing or point.value is None:
            return
        with output:
            clear_output(wait=True)
            show_preview(budget.value, three_way.value, point.value)

    def refresh(change=None):
        nonlocal refreshing
        selected_points, _ = coverage_preview(budget.value, three_way.value)
        previous = point.value
        refreshing = True
        try:
            point.options = [
                (f"{index}: M={row['M']}, K={row['K']}, N={row['N']}", index)
                for index, row in enumerate(selected_points)
            ]
            point.value = previous if previous in range(len(selected_points)) else 0
        finally:
            refreshing = False
        render()

    budget.observe(refresh, names="value")
    three_way.observe(refresh, names="value")
    point.observe(render, names="value")
    refresh()
    return widgets.VBox([widgets.HBox([budget, three_way]), point, output])


if widgets is None:
    show_preview()
else:
    display(coverage_widget())


## 5. Continue through Phase 0

A real run begins with `ComponentCoveragePlan.load`, which binds reviewed
operation/effect owners, budgets and roles to exact software/numerical/hardware
inputs. `component_coverage_inputs.expand` derives entries and resource boundaries.
`generation.generate_target` uses the ordinary writer to check bounds and write
programs, declarations, full-output goldens, coverage files and the capsule README.
`component_coverage.finalize` reopens those members and keeps missing obligations
visible.

The actual cohort names are printed below. Development entries are public tuning
inputs; functional guards are public checks with their declared mandatory status;
withheld transfer entries remain operator-only. Choose this split before tuning.
The public coverage summary exposes counts and commitments, not held-out selectors.

**Scope:** this notebook builds declarations, interface IR and reference values,
not an admitted corpus or executable result. It shares Merlin's integer arithmetic
and input materialization; other numerical formats need matching references.
Actual compilation, complete-output comparisons, instruction-use checks and held-out
quality evaluation remain separate. Phase 1 transcript conformance checks authoring
evidence, not numerical execution. Use the guide's public CLI with real selected
inputs and a fresh `out/` destination to continue.


In [ ]:
print("Actual component-plan cohorts:", COHORTS)
print("Preview status: declarations and expected values only; executable result unavailable.")


**Step 2 of 5: Test generation** | [All tutorials](README.md) | [Previous: Contracts and facts](01_contracts.ipynb) | [Next: Scheduling](03_scheduling.ipynb)
